[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/sandbox/gnina_docking_screen.ipynb)

# Docking screen with gnina or smina

**Sandbox notebook - not workshop material.**

The question this notebook answers is: *can we take a receptor-ligand complex, reproduce the
ligand's pose with docking, and then screen a small library in the same pocket?* It follows the
authors' own [gnina Colab](https://colab.research.google.com/drive/1QYo5QLUE80N_G28PlpYs6OKGddhhd931)
(download the binary, split a PDB with `grep`, redock with `--autobox_ligand`, view with py3Dmol),
with the test case swapped for a complex of our own and a library screen added on the end.

It was written with the blue group's CpABC1-silymarin complex in mind
(`Projects/BlueTeam/Data/CpABC1-Silymarin.pdb` in the shared Drive), but any PDB file holding a
protein with one bound molecule works.

**gnina** reports the classical Vina affinity plus two scores from a convolutional neural network
([Ragoza et al., J. Chem. Inf. Model. 2017](https://doi.org/10.1021/acs.jcim.6b00740),
[repository](https://github.com/gnina/gnina), Apache 2.0), and needs a Linux machine with an NVIDIA
GPU. Without one the notebook falls back to **smina**, the program gnina was forked from
([Koes et al., J. Chem. Inf. Model. 2013](https://doi.org/10.1021/ci300604z),
[repository](https://github.com/mwojcikowski/smina), Apache 2.0): same flags, CPU only, no neural
network scores. So this runs on a Colab GPU runtime and on a laptop.

## What you will do

- Pick a docking program: gnina if there is a GPU, smina if not
- Load a receptor-ligand complex and split it into its two parts
- Redock the reference ligand and measure how far the pose moved
- Dock a small library of silymarin analogues into the same pocket
- Rank the compounds and look at the best one

On Colab, set the runtime to **T4 GPU** (*Runtime > Change runtime type*) if you want the neural
network scores. Most of the time goes into the library screen either way.

## 1. Get a docking program

First the Python packages. Colab ships none of them; locally the notebook uses whatever is already
in the kernel you picked.

In [ ]:
import glob, os, platform, shutil, stat, subprocess, sys, urllib.request

IN_COLAB = "google.colab" in sys.modules
WORK = "work"  # everything this notebook produces goes here
os.makedirs(WORK, exist_ok=True)

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit", "py3Dmol", "stylia"], check=True)
else:
    print("Local run: using the packages already in this kernel.")

Then work out which docking program this machine can run. gnina needs Linux on x86-64 with a
working NVIDIA driver; anything else, including any Mac, means smina.

In [ ]:
linux_gpu = ((platform.system(), platform.machine()) == ("Linux", "x86_64")
             and shutil.which("nvidia-smi") is not None
             and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0)
ENGINE = "gnina" if linux_gpu else "smina"
print(f"{platform.system()} {platform.machine()} | Python {sys.version.split()[0]} | CUDA GPU: {'yes' if linux_gpu else 'no'} | engine: {ENGINE}")

gnina comes as a single 1.4 GB binary that needs no installation, but it was built against CUDA 12
and Colab has moved to CUDA 13. Most CUDA 12 libraries are still in the image, as leftovers of older
PyTorch wheels, but `libnvToolsExt.so.1` is not, so we fetch it into a folder of its own and point
`LD_LIBRARY_PATH` at every CUDA 12 library we can find. Nothing in the Python environment is touched.

smina, by contrast, is a half-megabyte CPU program from conda-forge, so there the cell only has to
find it on `PATH`. If it is not there:
`conda install -c conda-forge smina`, or `micromamba create -n smina -c conda-forge smina` if your
conda solver is slow, and then put that environment's `bin` on `PATH`.

> **Note:** if gnina reports `error while loading shared libraries: libsomething.so.12`, Colab has
> dropped another one. Add its `nvidia-<something>-cu12` wheel to the same `--target` install.

In [ ]:
ENV = dict(os.environ)

if ENGINE == "gnina":
    DOCK_BIN = f"{WORK}/gnina"
    if not os.path.exists(DOCK_BIN):
        urllib.request.urlretrieve("https://github.com/gnina/gnina/releases/download/v1.3.2/gnina.1.3.2", DOCK_BIN)
        os.chmod(DOCK_BIN, os.stat(DOCK_BIN).st_mode | stat.S_IEXEC)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--target", f"{WORK}/cuda12",
                        "nvidia-nvtx-cu12==12.1.105"], check=True)
    libs = glob.glob("/usr/local/lib/python3.*/dist-packages/nvidia/*/lib/*.so*") + glob.glob(f"{WORK}/cuda12/nvidia/*/lib/*.so*")
    ENV["LD_LIBRARY_PATH"] = ":".join(sorted({os.path.abspath(os.path.dirname(p)) for p in libs}))
else:
    DOCK_BIN = shutil.which("smina")
    assert DOCK_BIN, "No GPU here and no smina on PATH. Install it with: conda install -c conda-forge smina"

print(subprocess.run([DOCK_BIN, "--version"], capture_output=True, text=True, env=ENV).stdout.strip())

## 2. Load the complex

Docking needs a protein (the *receptor*) and a molecule already sitting in the pocket (the
*reference ligand*), which is what says where to search. Both come from one PDB file.

In Colab, run the cell, click **Choose Files** and pick the complex; it stays in that session only.
Running anywhere else, the cell reads `COMPLEX_PATH` instead, which points at the copy in the
developer's clone of this repository. That file is **not** committed: the repository is public and
the complex is the group's own unpublished work, so ask them for it, or take it from
`Projects/BlueTeam/Data` in the shared Drive.

In [ ]:
COMPLEX_PATH = "../projects/blue/data/cpabc1_silymarin.pdb"  # used when not running in Colab

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    complex_path = f"{WORK}/{name}"
    with open(complex_path, "wb") as f:
        f.write(uploaded[name])
else:
    complex_path = COMPLEX_PATH
print(f"{complex_path} | {os.path.getsize(complex_path) / 1e6:.1f} MB")

## 3. Split the complex

`ATOM` lines are the protein, `HETATM` lines are everything else: the bound molecule, but also water
and ions. The largest group of `HETATM` lines sharing one residue is taken as the ligand.

In [ ]:
lines = open(complex_path).read().splitlines()
protein = [line for line in lines if line.startswith("ATOM")]

residues = {}
for line in lines:
    if line.startswith("HETATM") and line[17:20].strip() not in ("HOH", "WAT"):
        residues.setdefault(line[17:26], []).append(line)  # residue name, chain and number

residue, ligand_lines = max(residues.items(), key=lambda item: len(item[1]))
with open(f"{WORK}/receptor.pdb", "w") as f:
    f.write("\n".join(protein) + "\nEND\n")
print(f"receptor: {len(protein)} atoms | ligand ({residue.strip()}): {len(ligand_lines)} atoms")

A PDB file records atom positions but not bonds, so RDKit works them out from the distances. The
SMILES string is the quick check that the right thing came out: for the CpABC1 complex it should be
silybin, the main component of silymarin.

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdDetermineBonds

ligand = Chem.MolFromPDBBlock("\n".join(ligand_lines), removeHs=False)
rdDetermineBonds.DetermineBondOrders(ligand, charge=0)
Chem.MolToMolFile(ligand, f"{WORK}/ligand.sdf")
print(Chem.MolToSmiles(Chem.RemoveHs(ligand)))

## 4. Redock the reference ligand

Take the ligand out, search for its position again, and see whether it comes back. `--autobox_ligand`
draws the search box around the reference ligand, `--seed 0` fixes the random numbers. gnina inherited
its command line from smina, so the same call works for both.

> **Note:** the seed does not make a run exactly repeatable. The search runs on several CPU threads
> and the order they finish in varies, so two identical runs of this notebook gave scores up to 0.6
> kcal/mol apart. Section 8 comes back to what that means for the ranking.

`EXHAUSTIVENESS` is how hard the search works. The search runs on the CPU in both programs, so what
sets the pace is how many cores you have. With 8, a molecule the size of silybin takes about 210
seconds on Colab's two vCPUs and about 10 seconds on a six-core laptop. Dropping to 4 on Colab takes
140 seconds and found the same best pose, but keep one value for the whole notebook so the scores
stay comparable.

In [ ]:
EXHAUSTIVENESS = 8

def dock(ligand_file, tag):
    """Dock one ligand into the receptor, inside the box around the reference ligand."""
    out_file = f"{WORK}/{tag}_docked.sdf"
    result = subprocess.run(
        [DOCK_BIN, "-r", f"{WORK}/receptor.pdb", "-l", ligand_file, "--autobox_ligand", f"{WORK}/ligand.sdf",
         "-o", out_file, "--seed", "0", "--exhaustiveness", str(EXHAUSTIVENESS)],
        capture_output=True, text=True, env=ENV)
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-2000:])
    return out_file

dock(f"{WORK}/ligand.sdf", "reference")

Every pose carries `affinity`, the Vina binding energy in kcal/mol, where more negative is better.
gnina adds two more: `cnn_pose_score`, the network's confidence that the pose is right, from 0 to 1,
and `cnn_affinity`, its binding-strength estimate as a pK value, where higher is better. Under gnina
the poses come out ordered by `cnn_pose_score`, under smina by affinity.

In [ ]:
import pandas as pd
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.warning")  # the output files are fine, RDKit just complains about them

def read_scores(docked_file):
    """Read the scores attached to each pose; the CNN columns exist only if gnina wrote the file."""
    rows = []
    for mol in Chem.SDMolSupplier(docked_file):
        if mol is None:
            continue
        row = {"name": mol.GetProp("_Name") or "ligand", "affinity": float(mol.GetProp("minimizedAffinity"))}
        if mol.HasProp("CNNscore"):
            row["cnn_pose_score"] = float(mol.GetProp("CNNscore"))
            row["cnn_affinity"] = float(mol.GetProp("CNNaffinity"))
        rows.append(row)
    return pd.DataFrame(rows)

read_scores(f"{WORK}/reference_docked.sdf")

## 5. Check the redocked pose

RMSD is the average distance between matching atoms of two poses, in angstroms; below 2 they count as
the same pose. Measure it for the best pose by affinity, and, when gnina wrote the file, for the pose
the network ranks first as well.

In [ ]:
from rdkit.Chem import rdMolAlign

def best_pose(docked_file, by="affinity"):
    """Return the best pose in an output file, by Vina affinity or by CNN pose score."""
    poses = [mol for mol in Chem.SDMolSupplier(docked_file) if mol is not None]
    if by == "affinity":
        return min(poses, key=lambda mol: float(mol.GetProp("minimizedAffinity")))
    return max(poses, key=lambda mol: float(mol.GetProp("CNNscore")))

reference = Chem.MolFromMolFile(f"{WORK}/ligand.sdf")
scores = read_scores(f"{WORK}/reference_docked.sdf")
for by in [c for c in ("affinity", "cnn_pose_score") if c in scores.columns]:
    rmsd = rdMolAlign.CalcRMS(best_pose(f"{WORK}/reference_docked.sdf", by), reference)
    print(f"best pose by {by:<15} RMSD {rmsd:5.2f} angstrom")
Chem.MolToMolFile(best_pose(f"{WORK}/reference_docked.sdf"), f"{WORK}/reference_best.sdf")

On the CpABC1 complex the pose with the best affinity comes back on top of the uploaded one: 0.50
angstrom under gnina, 0.75 under smina, with the same energy to two decimal places (-10.92 and
-10.91 kcal/mol). The search works.

The gnina run also showed why this check matters. The pose its network ranked first was 10.6
angstrom away, five places below the right one in its own ordering. That is what you would expect
from a network trained on crystal structures and applied to a computed model. **The ranking below
therefore uses the Vina affinity**, the one score both programs produce, with the CNN columns kept
as extra information when they exist.

This is the check worth doing on any new target before trusting a score.

Look at the two poses. Drawing every protein atom is slow, so only the residues lining the pocket
are kept, as thin sticks. The uploaded pose is grey, the redocked one green.

In [ ]:
import numpy as np

centre = reference.GetConformer().GetPositions().mean(axis=0)
coords = np.array([[float(line[30:38]), float(line[38:46]), float(line[46:54])] for line in protein])
pocket = [line for line, near in zip(protein, np.linalg.norm(coords - centre, axis=1) < 12) if near]
print(f"{len(pocket)} atoms within 12 angstrom of the ligand")

The viewer is interactive: drag to rotate, scroll to zoom.

In [ ]:
import py3Dmol

def view_pose(pose_file, reference_file=None):
    """Show a docked pose (green) in the pocket, next to a reference pose (grey)."""
    view = py3Dmol.view(width=700, height=450)
    view.addModel("\n".join(pocket), "pdb")
    view.setStyle({"stick": {"colorscheme": "whiteCarbon", "radius": 0.08}})
    if reference_file:
        view.addModel(open(reference_file).read(), "sdf")
        view.setStyle({"model": 1}, {"stick": {"colorscheme": "greyCarbon"}})
    pose_model = 2 if reference_file else 1
    view.addModel(open(pose_file).read(), "sdf")
    view.setStyle({"model": pose_model}, {"stick": {"colorscheme": "greenCarbon"}})
    view.zoomTo({"model": pose_model})
    return view

view_pose(f"{WORK}/reference_best.sdf", f"{WORK}/ligand.sdf")

## 6. Build the compound library

`silymarin_analogues.csv` next to this notebook holds sixteen public compounds related to silymarin:
the other flavonolignans of the extract and simpler flavonoids sharing the same core, with their
PubChem CIDs. The first ten are a deliberate mix of large and small, and those are the ones screened
by default. Any CSV with `name` and `smiles` columns works instead.

The reference ligand joins the library as well. It was docked already in section 4, but starting
from its own bound shape, an advantage none of the analogues get: rebuilt from a SMILES string it
scores -10.27 rather than -10.91 under smina, and -7.1 rather than -10.9 under gnina. Docking it
again the same way as the rest gives a number the analogues can fairly be compared with.

In [ ]:
LIBRARY_SIZE = 10

urllib.request.urlretrieve("https://raw.githubusercontent.com/ersilia-os/ub-cedd-projects-workshop/main/sandbox/silymarin_analogues.csv", f"{WORK}/silymarin_analogues.csv")
library = pd.read_csv(f"{WORK}/silymarin_analogues.csv").head(LIBRARY_SIZE)

reference_row = pd.DataFrame([{"name": "reference", "smiles": Chem.MolToSmiles(Chem.RemoveHs(ligand))}])
library = pd.concat([reference_row, library], ignore_index=True)
library[["name", "smiles"]]

A SMILES string carries no coordinates, so RDKit builds one 3D shape per compound and relaxes it
with a force field. One shape each is the cheap choice; a real campaign would generate several, since
the docking program only explores the rotatable bonds of the shape it is given.

In [ ]:
from rdkit.Chem import AllChem

def embed(smiles, name):
    """Turn a SMILES string into a single relaxed 3D structure."""
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)
    mol.SetProp("_Name", name)
    return mol

molecules = [embed(s, n) for n, s in zip(library["name"], library["smiles"])]
print(f"{len(molecules)} molecules with 3D coordinates")

## 7. Dock the library

Every compound goes through the same command, into the same box. On Colab's two vCPUs the big
flexible molecules take around 210 seconds and the small rigid ones around 110, so eleven compounds
are roughly half an hour. On a six-core laptop the same eleven take under two minutes. Each result
prints as it arrives.

> **Note:** on Colab, keep the browser tab awake. Output is buffered while the tab is disconnected,
> so a sleeping laptop makes the run look stuck when it is not.

In [ ]:
import time

rows = []
for i, mol in enumerate(molecules, start=1):
    name = mol.GetProp("_Name")
    tag = "lib_" + name.replace(" ", "_")
    Chem.MolToMolFile(mol, f"{WORK}/{tag}.sdf")
    start = time.time()
    best = read_scores(dock(f"{WORK}/{tag}.sdf", tag)).sort_values("affinity").iloc[0]
    rows.append({**best.to_dict(), "name": name})  # keeps the CNN columns when gnina wrote them
    print(f"{i:2d}/{len(molecules)}  {name:<20} affinity {best.affinity:6.2f}  ({time.time() - start:.0f} s)")

results = pd.DataFrame(rows)

## 8. Rank the compounds

Ranked by Vina affinity, most negative first, against the reference ligand docked the same way.

> **Note:** these are predictions on a computed receptor, from one conformer per compound and one
> docking run each. They are a way to order compounds for a closer look, nothing more.

In [ ]:
reference_affinity = results.loc[results["name"] == "reference", "affinity"].iloc[0]

ranked = results.sort_values("affinity").reset_index(drop=True)
ranked["beats_reference"] = ranked["affinity"] < reference_affinity
ranked.to_csv(f"{WORK}/docking_scores.csv", index=False)
print(f"reference: {reference_affinity:.2f} | compounds that beat it: {int(ranked['beats_reference'].sum())}")
ranked

Before reading anything into the order, look at how much of it is noise. There are two measurements
of that here, and both say about the same thing:

- The reference ligand and the two silybins are the same molecule. RDKit read the reference's
  stereocentres off the coordinates and they came out as neither of the forms PubChem lists, but the
  skeleton is silybin's. Those three rows still span 1.2 kcal/mol, from -10.27 to -11.43.
- Running the whole notebook twice, unchanged, moved individual compounds by up to 0.6 kcal/mol and
  swapped neighbouring rows.

So treat a difference of about a kcal/mol as nothing at all. Only a compound that stands clear of
the pack is worth a second look, and the honest way to use this list is as a shortlist, not a
ranking.

The same numbers as a plot, with the reference drawn as a horizontal line. The bars point downwards
because a binding energy is negative, so the ones reaching furthest down are the best.

In [ ]:
import stylia

stylia.set_format("slide")
stylia.set_style("ersilia")
colors = stylia.NamedColors()

fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.bar(ranked["name"], ranked["affinity"],
       color=[colors.plum if b else colors.gray for b in ranked["beats_reference"]])
ax.axhline(reference_affinity, color=colors.blue)
ax.tick_params(axis="x", rotation=90)
stylia.label(ax, xlabel="", ylabel="Affinity (kcal/mol)")
stylia.save_figure(f"{WORK}/docking_scores.png")

Finally the best analogue in the pocket, with the uploaded ligand behind it in grey.

In [ ]:
best = ranked[ranked["name"] != "reference"].iloc[0]
top_pose = best_pose(f"{WORK}/lib_{best['name'].replace(' ', '_')}_docked.sdf")
Chem.MolToMolFile(top_pose, f"{WORK}/top_hit.sdf")
print(f"Best analogue: {best['name']} (affinity {best['affinity']:.2f} kcal/mol)")
view_pose(f"{WORK}/top_hit.sdf", f"{WORK}/ligand.sdf")

## Summary

- Both programs work. On the CpABC1 complex they agree on the redocked silymarin to two decimal
  places (-10.92 gnina, -10.91 smina) and both put it back on the crystal-like pose, 0.50 and 0.75
  angstrom away. If gnina will not start at all, the CUDA 12 libraries in section 1 are the first
  place to look.
- Section 5 is the useful lesson: on this computed receptor gnina's CNN ranking put the correct pose
  fifth, so redocking is what tells you which score to trust before ranking anything.
- Section 8 is the second one: the same molecule, as the reference and as the two silybins, spans
  1.2 kcal/mol, and repeating the notebook moves compounds by up to 0.6. That is the size of the
  noise, and it is most of the gap between neighbouring rows.
- Results are in `work/docking_scores.csv` and `work/docking_scores.png`. On Colab both are wiped
  with the session.

Rough edges worth knowing:

- The gnina binary is 1.4 GB and built against CUDA 12, so it needs the library shim in section 1 on
  today's Colab image. smina is half a megabyte and needs none of that, but gives no CNN scores.
- The search runs on the CPU in both programs, so the GPU only pays for the CNN rescoring. Eleven
  compounds cost about half an hour on Colab's two cores and under two minutes on a six-core laptop.
- No protonation states, one conformer per compound, no receptor flexibility.

**Next:** if this earns a place in the blue group's project, the version for participants would read
the complex from `projects/blue/data/` rather than an upload, and the screen would have to be short
enough to sit inside a workshop session.